# Project 12 · Self-Hosted Agent Service, Live on a Public URL: the release review (Starter notebook)

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 6–10 hours (notebook 2–3 h; the deployment and its write-up the rest) |
| 📋 Prerequisites | Lessons 12.01–12.09 |
| 🔑 Keys | none for the notebook · a GitHub, Render and Neon account and `GROQ_API_KEY` for the public deployment |
| 💸 Cost | $0 offline; the public demo uses your Groq free tier, capped by `BUDGET_GLOBAL_DAY_USD` |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |
| 📄 Spec | [project/README.md](README.md) |

**The job:** Orbita Cloud wants the support desk on a public URL. Before anyone gets the link, you run the release review a production team would: prove tenant isolation on every endpoint, idempotency and the double-texting policy, a resumable stream that delivers each event exactly once, data that survives a redeploy, a complete `DELETE /v1/me/data`, 20 concurrent users against the SLOs, the eval gate, and then write the SHIP/HOLD report your lead signs.

This is the **starter**. Every `# TODO` marks work for you. The notebook runs top to bottom even before you finish: unfinished parts show ❌ in the checks instead of crashing. Stuck? Each TODO has a collapsed hint; the [reference solution](solutions/12-project-production-agent-service-solution.ipynb) is the last resort.

## Requirements (from the spec)

1. **The service** is `apps/m12_service` (Lessons 12.04–12.09): versioned API under `/v1`, Postgres or SQLite state, API keys or JWT, budgets, UI, image, Compose, metrics, alerts, runbook, CI/CD, public deployment.
2. **Evidence:** each requirement is checked by code against a running instance, not by reading the code.
3. **Report:** a Markdown release review with a SHIP/HOLD decision that follows the bar: every acceptance check passes, the SLOs hold under 20 concurrent users, and the eval gate passes.

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `isolation_suite(c, thread_id, run_id)`: another user and no key, on every endpoint |
| 2 | Step 3 | `exactly_once(c, run_id)`: disconnect, resume with `Last-Event-ID` |
| 3 | Step 4 | `rows_for(db, user, threads, runs)`: count a user's rows in every table |
| 4 | Step 3 | `policy_check(c)`: idempotency key and double texting |
| 5 | Step 8 | `write_report(path, evidence)` |

---
## Step 0 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-project'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Step 1 · The service under review

The same app the image runs (`create_app(durable_support_manager)`), in DEMO mode with 0.2 s per offline model call, on a SQLite file in `data/outputs/m12/` so that a restart keeps its data, exactly like Postgres in Compose. The client talks to `/v1`.

In [ ]:
# The service under review: apps/m12_service's app on a SQLite file, so "restart" keeps its data.
import json
import sqlite3
import subprocess
import time

import httpx

from mastery.service.app import create_app, serve_in_background
from mastery.service.client import stream_events
from mastery.service.desk import durable_support_manager
from mastery.service.loadtest import run_load
from mastery.service.settings import ServiceSettings

OUT_DIR = REPO_ROOT / "data" / "outputs" / "m12"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DB = OUT_DIR / "project12.db"
for leftover in OUT_DIR.glob("project12.db*"):
    leftover.unlink()                                  # a fresh database for every run of this notebook
KEYS = {"customer": "demo-customer", "other": "demo-other", "manager": "demo-manager"}   # DEMO keys: local only
USERS = {"customer": "C-1002", "other": "C-1001", "manager": "ops"}


def h(who):
    return {"Authorization": f"Bearer {KEYS[who]}"}


def start_service(release="1.0.0"):
    settings = ServiceSettings(database_url=f"sqlite:///{DB.as_posix()}", release=release, log_level="ERROR",
                               max_concurrent=20)   # the offline model is cheap; LIVE on a free instance keeps 2
    base, stop = serve_in_background(create_app(lambda s: durable_support_manager(s, delay_s=0.2), settings))
    return base, stop, httpx.Client(base_url=f"{base}/v1", timeout=60)


def wait_done(c, run_id, who="customer"):
    return stream_events(c, run_id, KEYS[who])[-1]


BASE, stop_service, c = start_service()
FAQ = "How do I rotate an API key?"
REFUND = "I'm customer C-1002. I want a refund of $199 for INV-2001, we moved to another vendor."
thread = c.post("/threads", headers=h("customer")).json()["thread_id"]
run = c.post(f"/threads/{thread}/runs", json={"message": FAQ}, headers=h("customer")).json()
print("service :", BASE, "·", httpx.get(f"{BASE}/version").json())
print("first run:", wait_done(c, run["run_id"]).event, "· history:", len(c.get(f"/threads/{thread}", headers=h("customer")).json()["messages"]), "messages")

**You should now see** the local URL with `{'release': '1.0.0', 'mode': 'demo', …}`, then `first run: end · history: 2 messages`.

---
## Step 2 · Tenant isolation, endpoint by endpoint

**TODO 1** is in the cell below.

> 💡 **Hint:** Return one row per probe. Probe the thread's history, start-a-run, delete, and the run's status, stream and cancel, first with **another customer's** key (expect 404: not 403, so you don't reveal it exists) and then **without** a key (expect 401). Add one row for the other customer's thread list. `c.request(method, path, json=..., headers=...)` sends any method.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def isolation_suite(c, thread_id, run_id):
    """One row per probe: {"check", "status", "expected", "ok"}. Probe every endpoint of the thread and run."""
    probes = [("GET", f"/threads/{thread_id}", None), ("POST", f"/threads/{thread_id}/runs", {"message": "hi"}),
              ("DELETE", f"/threads/{thread_id}", None), ("GET", f"/runs/{run_id}", None),
              ("GET", f"/runs/{run_id}/stream", None), ("POST", f"/runs/{run_id}/cancel", None)]
    rows = []
    for who, expected in (("other", 404), (None, 401)):
        for method, path, body in probes:
            response = c.request(method, path, json=body, headers=h(who) if who else {})
            rows.append({"check": f"{who or 'no key'} {method} {path.split('/')[1]}{'/…' * (path.count('/') - 1)}",
                         "status": response.status_code, "expected": expected, "ok": response.status_code == expected})
    listed = thread_id in c.get("/threads", headers=h("other")).json()["threads"]
    rows.append({"check": "other GET /threads lists it", "status": listed, "expected": False, "ok": not listed})
    return rows
```

</details>

In [ ]:
def isolation_suite(c, thread_id, run_id):
    """One row per probe: {"check", "status", "expected", "ok"}. Probe every endpoint of the thread and run."""
    rows = []
    # TODO 1: call each thread and run endpoint with another customer's key, and without a key
    return rows

In [ ]:
rows = isolation_suite(c, thread, run["run_id"])
table(rows, ["check", "status", "expected", "ok"]) if rows else print("❌ no probes yet (TODO 1)")
print("thread still there:", c.get(f"/threads/{thread}", headers=h("customer")).status_code == 200)

**You should now see** (solution) 13 rows, all `True`: 404 for the other customer on each of the six endpoints, 401 without a key, and the thread missing from the other customer's list; then `thread still there: True` (their `DELETE` didn't delete it).

---
## Step 3 · Streams, idempotency and double texting

**TODO 2** is in the cell below.

> 💡 **Hint:** `stream_events(c, run_id, key, limit=n)` stops after n events (a disconnect); `last_event_id=` reconnects. Exactly once means: the ids before plus the ids after equal a fresh full read, with no id twice.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def exactly_once(c, run_id, cut=2):
    """Read `cut` events, disconnect, reconnect with Last-Event-ID. Return the ids and whether nothing was lost or repeated."""
    key = KEYS["customer"]
    before = [e.id for e in stream_events(c, run_id, key, limit=cut)]
    after = [e.id for e in stream_events(c, run_id, key, last_event_id=before[-1])]
    full = [e.id for e in stream_events(c, run_id, key)]
    return {"before": before, "after": after, "ok": before + after == full and len(set(full)) == len(full)}
```

</details>

**TODO 4** is in the cell below.

> 💡 **Hint:** Send the same message twice with the same `Idempotency-Key` header: both answers must name the same `run_id`. Then, while that run is still going (each offline model call takes 0.2 s), send a different message on the same thread without a key: the service's policy is *reject*, so expect 409. Wait for the first run before returning.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def policy_check(c):
    """The double-texting and idempotency policy, tested: {"idempotent": bool, "double_text": status code}."""
    t = c.post("/threads", headers=h("customer")).json()["thread_id"]
    headers = h("customer") | {"Idempotency-Key": "project12-1"}
    first = c.post(f"/threads/{t}/runs", json={"message": FAQ}, headers=headers).json()
    again = c.post(f"/threads/{t}/runs", json={"message": FAQ}, headers=headers).json()
    second = c.post(f"/threads/{t}/runs", json={"message": "and my invoices?"}, headers=h("customer"))
    wait_done(c, first["run_id"])
    return {"idempotent": first["run_id"] == again["run_id"], "double_text": second.status_code}
```

</details>

In [ ]:
def exactly_once(c, run_id, cut=2):
    """Read `cut` events, disconnect, reconnect with Last-Event-ID. Return the ids and whether nothing was lost or repeated."""
    # TODO 2: before / after / a full read for comparison
    return {"before": [], "after": [], "ok": False}

In [ ]:
def policy_check(c):
    """The double-texting and idempotency policy, tested: {"idempotent": bool, "double_text": status code}."""
    # TODO 4: same Idempotency-Key twice; a second message while the first run is still running
    return {"idempotent": False, "double_text": None}

In [ ]:
slow = c.post(f"/threads/{thread}/runs", json={"message": "Show my invoices"}, headers=h("customer")).json()
once = exactly_once(c, slow["run_id"])
print("before:", once["before"], "· after:", once["after"][:3], "…", "· exactly once:", "✅" if once["ok"] else "❌")
policy = policy_check(c)
print("idempotent:", policy["idempotent"], "· double text →", policy["double_text"], "(policy: reject = 409)")

**You should now see** (solution) `before: ['0', '1']`, the ids after it starting at `'2'`, `exactly once: ✅`, then `idempotent: True · double text → 409`.

---
## Step 4 · A redeploy keeps the data

**TODO 3** is in the cell below.

> 💡 **Hint:** Open the file with `sqlite3.connect` and `COUNT(*)` each table: `svc_threads` (owner), `svc_runs` and `svc_idempotency` (user_id), `svc_events` (by the run ids you pass in) and LangGraph's `checkpoints` and `writes` (by thread id). Use `?` placeholders, never f-strings with ids.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def rows_for(db_path, user_id, thread_ids, run_ids):
    """Rows per table that belong to this user, in the service tables AND LangGraph's checkpoint tables."""
    def count(sql, values):
        if not values:
            return 0
        marks = ",".join("?" * len(values))
        return db.execute(sql.format(marks=marks), list(values)).fetchone()[0]

    with sqlite3.connect(db_path) as db:
        return {
            "svc_threads": db.execute("SELECT COUNT(*) FROM svc_threads WHERE owner = ?", (user_id,)).fetchone()[0],
            "svc_runs": db.execute("SELECT COUNT(*) FROM svc_runs WHERE user_id = ?", (user_id,)).fetchone()[0],
            "svc_events": count("SELECT COUNT(*) FROM svc_events WHERE run_id IN ({marks})", run_ids),
            "svc_idempotency": db.execute("SELECT COUNT(*) FROM svc_idempotency WHERE user_id = ?", (user_id,)).fetchone()[0],
            "checkpoints": count("SELECT COUNT(*) FROM checkpoints WHERE thread_id IN ({marks})", thread_ids),
            "writes": count("SELECT COUNT(*) FROM writes WHERE thread_id IN ({marks})", thread_ids),
        }
```

</details>

In [ ]:
def rows_for(db_path, user_id, thread_ids, run_ids):
    """Rows per table that belong to this user, in the service tables AND LangGraph's checkpoint tables."""
    # TODO 3: count with SQL; ids are passed in because deleted runs can't be looked up by user any more
    return {}

In [ ]:
# A paused refund, then a "deploy": stop the process, start a new one on the same database.
refund_thread = c.post("/threads", headers=h("customer")).json()["thread_id"]
refund = c.post(f"/threads/{refund_thread}/runs", json={"message": REFUND}, headers=h("customer")).json()
print("before restart:", wait_done(c, refund["run_id"]).event)
my_threads = c.get("/threads", headers=h("customer")).json()["threads"]
my_runs = [r[0] for r in sqlite3.connect(DB).execute("SELECT run_id FROM svc_runs WHERE user_id = ?", (USERS["customer"],))]
counts_before = rows_for(DB, USERS["customer"], my_threads, my_runs)
c.close()
stop_service()
BASE, stop_service, c = start_service(release="1.0.1")
counts_after = rows_for(DB, USERS["customer"], my_threads, my_runs)     # the new process, before any new work
print("after restart :", c.get(f"/threads/{refund_thread}", headers=h("customer")).json()["waiting_for_approval"], "waiting for approval")
resumed = c.post(f"/threads/{refund_thread}/runs", json={"resume": "approve"}, headers=h("manager")).json()
print("approved      :", wait_done(c, resumed["run_id"], "manager").data["answer"][:60])
table([{"table": k, "before": v, "after restart": counts_after.get(k)} for k, v in counts_before.items()],
      ["table", "before", "after restart"]) if counts_before else print("❌ no counts yet (TODO 3)")

**You should now see** `before restart: interrupt`, `after restart : True waiting for approval`, the manager's `Refunded $199.00 …`, and a table where every count is above 0 and equal before and after the restart (solution). The new process (release `1.0.1`) found the paused refund in the checkpoint and resumed it.

---
## Step 5 · `DELETE /v1/me/data`

In [ ]:
other_thread = c.post("/threads", headers=h("other")).json()["thread_id"]
wait_done(c, c.post(f"/threads/{other_thread}/runs", json={"message": FAQ}, headers=h("other")).json()["run_id"], "other")
my_runs = [r[0] for r in sqlite3.connect(DB).execute("SELECT run_id FROM svc_runs WHERE user_id = ?", (USERS["customer"],))]
my_threads = c.get("/threads", headers=h("customer")).json()["threads"]
erased = c.delete("/me/data", headers=h("customer")).json()
mine_after = rows_for(DB, USERS["customer"], my_threads, my_runs)
theirs = rows_for(DB, USERS["other"], [other_thread], [r[0] for r in sqlite3.connect(DB).execute(
    "SELECT run_id FROM svc_runs WHERE user_id = ?", (USERS["other"],))])
print("DELETE /v1/me/data:", erased)
print("my rows left     :", mine_after or "❌ (TODO 3)")
print("other user's rows:", theirs or "❌ (TODO 3)")

**You should now see** (solution) the deleted counts, `my rows left` all 0 (service tables **and** checkpoints) and the other customer's rows untouched. Spend rows are not in the list on purpose: they hold no conversation and enforce the caps; the privacy note in your README should say so.

---
## Step 6 · 20 concurrent users against the SLOs, and the eval gate

In [ ]:
def spent():                                       # what the service charged (svc_spend: one row per run)
    return sqlite3.connect(DB).execute("SELECT COALESCE(SUM(usd), 0) FROM svc_spend").fetchone()[0]


spent_before = spent()
load = await run_load(f"{BASE}/v1", users=20)
cost = spent() - spent_before
print(f"{load['users']} users in {load['wall_s']} s · {load['by_status']}")
print(f"TTFT p50 {load['ttft_p50']} s · p95 {load['ttft_p95']} s · duration p95 {load['duration_p95']} s · cost ${cost:.4f}")
table(load["slos"], ["name", "ratio", "met"])

In [ ]:
gate = subprocess.run([sys.executable, "-m", "mastery.evals.release_gate"], capture_output=True, text=True,
                      encoding="utf-8", cwd=REPO_ROOT, env={**os.environ, "PYTHONPATH": str(SRC_DIR)})
print("eval gate exit:", gate.returncode, "·", gate.stdout.strip().splitlines()[-1])

**You should now see** `20 users … {'ok': 20, 'error': 0, …}`, TTFT and duration percentiles, TTFT p95 around 1 s, the three SLOs all `met` (this offline service runs 20 at once; a free LIVE instance keeps 2 and queues the rest, so measure yours), the load test's DEMO cost, and `eval gate exit: 0 · ✅ all gates passed`.

---
## Step 7 · 🌐 Optional: the public deployment

In [ ]:
# 🌐 Optional: your public deployment (Lesson 12.09). No model call.
PUBLIC_URL = os.environ.get("M12_PUBLIC_URL", "").strip().rstrip("/")
HAVE_PUBLIC_URL = bool(PUBLIC_URL)
public = "not checked (set M12_PUBLIC_URL)"
if HAVE_PUBLIC_URL:
    with httpx.Client(base_url=PUBLIC_URL, timeout=120) as pub:
        started = time.perf_counter()
        version = pub.get("/version").json()
        anonymous = pub.post("/v1/threads").status_code
        demo = pub.post("/v1/threads", headers=h("customer")).status_code
    public = f"{PUBLIC_URL} release {version['release']}, first response {time.perf_counter() - started:.0f} s, no key {anonymous}, demo key {demo}"
    print(public)
    if demo != 401:
        print("❌ the public deployment accepts the demo key: set MASTERY_SERVICE_PUBLIC=true and real keys")
else:
    print("⏭️ Skipped: set M12_PUBLIC_URL in .env to include your deployment in the review.")

**You should now see** your URL, release, the first response time (about a minute after a spin-down) and `no key 401, demo key 401`, or `⏭️ Skipped` without `M12_PUBLIC_URL`.

---
## Step 8 · The release review

**TODO 5** is in the cell below.

> 💡 **Hint:** Build the Markdown from `ev` only: sections `## Decision` (with **SHIP** or **HOLD** and the blockers), `## Acceptance` (one row per check), `## SLOs under load` (users, statuses, TTFT p50/p95, duration p95 and each SLO), `## Cost` and `## Risks and next steps`. Never type a number by hand.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def write_report(path, ev):
    """The Markdown release review: Decision, Acceptance, SLOs, Cost, Risks. Every number from `ev`."""
    load = ev["load"]
    ship = ev["decision"] == "SHIP"
    lines = [f"# Project 12 release review: {ev['release']}", "", "## Decision", "",
             f"**{ev['decision']}**: " + ("every acceptance check passed, the SLOs were met under load and the eval gate passed."
                                          if ship else "blocked by: " + ", ".join(ev["blockers"]) + "."), "",
             "## Acceptance", "", "| Check | Result |", "|---|---|"]
    lines += [f"| {name} | {'pass' if ok else 'FAIL'} |" for name, ok in ev["checks"].items()]
    lines += ["", "## SLOs under load", "",
              f"{load['users']} concurrent users in {load['wall_s']} s: {load['by_status']}. "
              f"TTFT p50 {load['ttft_p50']} s, p95 {load['ttft_p95']} s; duration p95 {load['duration_p95']} s.", "",
              "| SLO | Ratio | Met |", "|---|---|---|"]
    lines += [f"| {s['name']} | {s['ratio']} | {s['met']} |" for s in load["slos"]]
    lines += ["", "## Cost", "", f"Load test: ${ev['cost_usd']:.4f} in total, ${ev['cost_usd'] / max(1, load['users']):.5f} per run "
              "(DEMO prices: estimated from token counts of the offline model).", "",
              "## Risks and next steps", "",
              f"- Eval gate exit code {ev['gate_exit']}; public URL: {ev['public']}.",
              "- 20 users on one process is not a capacity test: repeat against the deployed URL, where cold starts add about a minute.",
              "- DEMO answers come from the offline model; run the 💸 check and the eval gate on the real model before inviting users.",
              "- Spend rows survive `DELETE /me/data` by design (they enforce the caps); say so in the privacy note."]
    text = "\n".join(lines) + "\n"
    path.write_text(text, encoding="utf-8")
    return text
```

</details>

In [ ]:
def write_report(path, ev):
    """The Markdown release review: Decision, Acceptance, SLOs, Cost, Risks. Every number from `ev`."""
    # TODO 5
    text = "# Project 12 release review\n\nTODO\n"
    path.write_text(text, encoding="utf-8")
    return text

In [ ]:
checks = {
    "tenant isolation (404/401 on every endpoint)": len(rows) >= 7 and all(r["ok"] for r in rows),
    "idempotency key + double-text policy (409)": policy["idempotent"] and policy["double_text"] == 409,
    "stream resumes exactly once": once["ok"] and bool(once["after"]),
    "data survives a restart": bool(counts_before) and counts_before == counts_after and min(counts_before.values()) > 0,
    "DELETE /me/data leaves no rows": bool(mine_after) and sum(mine_after.values()) == 0 and theirs.get("svc_threads", 0) > 0,
    "20 concurrent runs, no errors": load["by_status"]["error"] == 0 and load["by_status"]["ok"] == 20,
}
slos_met = all(s["met"] for s in load["slos"] if s["met"] is not None)
blockers = [name for name, ok in checks.items() if not ok] + ([] if slos_met else ["SLOs under load"]) + (
    [] if gate.returncode == 0 else ["eval gate"])
evidence = {"release": "1.0.1", "checks": checks, "load": {k: v for k, v in load.items() if k != "records"},
            "cost_usd": cost, "gate_exit": gate.returncode, "public": public,
            "decision": "HOLD" if blockers else "SHIP", "blockers": blockers}
report = write_report(OUT_DIR / "project12-report.md", evidence)
print(report[:900])

**You should now see** the report beginning `# Project 12 release review: 1.0.1`, `## Decision` and **SHIP** or **HOLD** with its blockers (solution: SHIP if the TTFT SLO held on your machine). Saved to `data/outputs/m12/project12-report.md`.

---
## Acceptance checks

In [ ]:
def accept(name, ok):
    results[name] = bool(ok)
    print(("✅" if ok else "❌"), name)

results = {}
for name, ok in checks.items():
    accept(name, ok)
accept("report: every section", all(s in report for s in ("## Decision", "## Acceptance", "## SLOs under load", "## Cost", "## Risks")))
accept("report: measured numbers and a decision that follows the bar",
       f"p95 {load['ttft_p95']} s" in report and f"**{evidence['decision']}**" in report
       and (evidence["decision"] == "SHIP") == (not blockers))
print(f"\n{sum(results.values())}/{len(results)} acceptance checks passed")
(OUT_DIR / "project12-acceptance.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
c.close()
stop_service()

**You should now see** `8/8 acceptance checks passed` (solution). Starter: `1/8` (the load test needs no TODO; everything else does). The checks run against a live process: the probes hit real endpoints, the row counts read the database directly, the restart is a real new process, and the report must carry the measured p95 and a decision that follows the bar. Results: `data/outputs/m12/project12-acceptance.json`.

## Findings (write yours)

- Which check would have caught a real incident you have seen (or read about)?
- Under load, which stage used the TTFT budget: the queue or the model? What would you change first?
- What does your public demo still lack before real customers use it (see 12.09's production notes)?

## ✅ Before you commit

- [ ] Restart & Run All passes without a key.
- [ ] All TODOs done; 8/8 acceptance checks; report and findings written.
- [ ] The public URL is in your README, behind real keys and the cost cap.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **Module 12 done:** your agent runs as a service you can operate, release safely and show on a public URL.

**Next:** see [COURSE_OUTLINE.md](../../../COURSE_OUTLINE.md) for the following module. Back to the [Module 12 README](../../../modules/12-production-and-deployment/README.md).